In [2]:
# Cell 1: Mount GDrive + Extract code
from google.colab import drive
drive.mount('/gdrive')

import os, sys, subprocess

# Verify thermalwatch structure
contents = os.listdir('/gdrive/MyDrive/thermalwatch/')
print(f'GDrive thermalwatch: {contents}')

# Extract code into /content/thermalwatch
os.makedirs('/content/thermalwatch', exist_ok=True)
subprocess.run([
    'tar', '-xzf',
    '/gdrive/MyDrive/thermalwatch/project/thermalwatch.tar.gz',
    '-C', '/content/thermalwatch/'  # extract INTO thermalwatch/
], check=True)

os.chdir('/content/thermalwatch')
sys.path.insert(0, '/content/thermalwatch')

print(f'\nContents: {os.listdir("/content/thermalwatch")}')
print('✅ Code extracted')

Mounted at /gdrive
GDrive thermalwatch: ['project', 'data', 'checkpoints']

Contents: ['scripts', 'notebooks', 'requirements.txt', 'src']
✅ Code extracted


In [3]:
# Cell 2: Install Python 3.11 + setup venv
import subprocess, os

# Install Python 3.11
os.system('apt-get install -y python3.11 python3.11-venv python3.11-dev -q')
os.system('apt-get install -y libgeos-dev libgdal-dev -q')
os.environ['MPLBACKEND'] = 'agg'

VENV_PYTHON = '/content/venv311/bin/python3'
VENV_PIP    = '/content/venv311/bin/pip'

# Create venv with Python 3.11
os.system('python3.11 -m venv /content/venv311')
os.system('curl -sS https://bootstrap.pypa.io/get-pip.py | /content/venv311/bin/python3')

def install(packages):
    r = subprocess.run(
        [VENV_PIP, 'install', '-q'] + packages,
        capture_output=True, text=True
    )
    if r.returncode != 0:
        print(f'Error: {r.stderr[-300:]}')
    return r.returncode == 0

print('Step 1: terratorch...')
install(['terratorch'])

print('Step 2: torch==2.2.0...')
install(['--force-reinstall', 'torch==2.2.0', 'torchvision==0.17.0'])

print('Step 3: dependencies...')
install([
    'transformers==4.40.0', 'timm', 'einops',
    'rasterio', 'geopandas', 'shapely', 'pyproj',
    'pystac-client', 'planetary-computer',
    'scipy', 'scikit-learn', 'netCDF4',
    'requests', 'pyyaml', 'tqdm', 'pandas',
    'huggingface_hub==0.20.3',
])

print('Step 4: pinning versions...')
install([
    'numpy==1.26.4',
    'protobuf==3.20.3',
    'setuptools==69.5.1',
])

# Verify
r = subprocess.run([VENV_PYTHON, '-c', '''
import os
os.environ["MPLBACKEND"] = "agg"
import numpy as np, torch
from terratorch.registry import BACKBONE_REGISTRY
print(f"python:     {__import__('sys').version}")
print(f"numpy:      {np.__version__}")
print(f"torch:      {torch.__version__}")
print(f"cuda:       {torch.cuda.is_available()}")
print(f"gpu:        {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print("terratorch: OK")
'''], capture_output=True, text=True)
print(r.stdout)
if r.stderr:
    print('STDERR:', r.stderr[-300:])

Step 1: terratorch...
Step 2: torch==2.2.0...
Step 3: dependencies...
Step 4: pinning versions...
python:     3.11.15 (main, Mar  3 2026, 09:26:23) [GCC 13.3.0]
numpy:      1.26.4
torch:      2.2.0+cu121
cuda:       True
gpu:        Tesla T4
terratorch: OK



In [3]:
# Cell 3: Verify Prithvi with correct input shape
import subprocess

VENV_PYTHON = '/content/venv311/bin/python3'

r = subprocess.run([VENV_PYTHON, '-c', '''
import os, torch
os.environ["MPLBACKEND"] = "agg"

from terratorch.registry import BACKBONE_REGISTRY

print("Loading Prithvi-EO-2.0-300M...")
encoder = BACKBONE_REGISTRY.build(
    "prithvi_eo_v2_300",
    pretrained=True,
    num_frames=1,
    in_chans=6,
)

params = sum(p.numel() for p in encoder.parameters())
print(f"✅ Loaded: {params/1e6:.0f}M parameters")

# Correct shape: [B, C, T, H, W] = [1, 6, 1, 224, 224]
x = torch.zeros(1, 6, 1, 224, 224)
with torch.no_grad():
    out = encoder(x)

feat = out[-1] if isinstance(out, (list, tuple)) else out
print(f"✅ Output shape: {feat.shape}")
emb  = feat[:, 1:, :].mean(1) if feat.dim() == 3 else feat.mean([2, 3])
print(f"✅ Embedding: {emb.shape}")

if torch.cuda.is_available():
    encoder = encoder.cuda()
    with torch.no_grad():
        out_gpu = encoder(x.cuda())
    feat_gpu = out_gpu[-1] if isinstance(out_gpu, (list, tuple)) else out_gpu
    emb_gpu  = feat_gpu[:, 1:, :].mean(1) if feat_gpu.dim() == 3 else feat_gpu.mean([2, 3])
    print(f"✅ GPU: {emb_gpu.shape} on {torch.cuda.get_device_name(0)}")

del encoder
torch.cuda.empty_cache()
print("✅ Prithvi ready")
'''], capture_output=True, text=True)
print(r.stdout)
if r.stderr:
    print('STDERR:', r.stderr[-300:])

Loading Prithvi-EO-2.0-300M...
✅ Loaded: 304M parameters
✅ Output shape: torch.Size([1, 197, 1024])
✅ Embedding: torch.Size([1, 1024])
✅ GPU: torch.Size([1, 1024]) on Tesla T4
✅ Prithvi ready

STDERR: hvi_EO_V2_300M.pt:  97%|█████████▋| 1.29G/1.33G [00:13<00:00, 77.1MB/s]
Prithvi_EO_V2_300M.pt:  99%|█████████▉| 1.31G/1.33G [00:14<00:00, 93.7MB/s]
Prithvi_EO_V2_300M.pt: 100%|██████████| 1.33G/1.33G [00:14<00:00, 105MB/s] 
Prithvi_EO_V2_300M.pt: 100%|██████████| 1.33G/1.33G [00:14<00:00, 93.6MB/s]



In [4]:
# Cell 3: Test full backbone
import subprocess, os, sys

VENV_PYTHON = '/content/venv311/bin/python3'

r = subprocess.run([VENV_PYTHON, '-c', '''
import os, sys, torch
os.environ["MPLBACKEND"] = "agg"
sys.path.insert(0, "/content/thermalwatch")

from src.models.backbone.thermal_backbone import ThermalWatchBackbone

print("Loading ThermalWatchBackbone...")
backbone = ThermalWatchBackbone(
    pretrained_optical=True,
    freeze_optical=True,
)
backbone = backbone.cuda()
backbone.eval()

B = 2
thermal = torch.randn(B, 1, 224, 224).cuda()
optical = torch.randn(B, 6, 224, 224).cuda()
weather = torch.randn(B, 5).cuda()
osm     = torch.randn(B, 7).cuda()

with torch.no_grad():
    emb = backbone(
        thermal=thermal,
        optical=optical,
        weather=weather,
        osm=osm,
    )

print(f"✅ With S2:    {emb.shape}")

with torch.no_grad():
    emb2 = backbone(thermal=thermal, osm=osm)
print(f"✅ Without S2: {emb2.shape}")

print("✅ Backbone ready for training")
'''], capture_output=True, text=True)
print(r.stdout)
if r.stderr:
    print('STDERR:', r.stderr[-500:])

Loading ThermalWatchBackbone...
✅ With S2:    torch.Size([2, 512])
✅ Without S2: torch.Size([2, 512])
✅ Backbone ready for training

STDERR: Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth

  0%|          | 0.00/44.7M [00:00<?, ?B/s]
 19%|█▉        | 8.50M/44.7M [00:00<00:00, 89.1MB/s]
 67%|██████▋   | 30.0M/44.7M [00:00<00:00, 169MB/s] 
100%|██████████| 44.7M/44.7M [00:00<00:00, 173MB/s]



In [7]:
# Cell 5: Copy data + verify datasets
import shutil, os, json, subprocess
from pathlib import Path

GDRIVE = '/gdrive/MyDrive/thermalwatch'
COLAB  = '/content/thermalwatch/data'

# Copy wildfire patches
print('Copying wildfire patches (~29GB)...')
shutil.copytree(
    f'{GDRIVE}/data/wildfire/patches',
    f'{COLAB}/wildfire/patches',
    dirs_exist_ok=True
)
print('✅ Wildfire patches copied')

# Copy solar patches
print('Copying solar patches (~19GB)...')
shutil.copytree(
    f'{GDRIVE}/data/solar/patches',
    f'{COLAB}/solar/patches',
    dirs_exist_ok=True
)
print('✅ Solar patches copied')

# Copy indexes
print('Copying indexes...')
shutil.copytree(
    f'{GDRIVE}/data/indexes',
    f'{COLAB}/indexes',
    dirs_exist_ok=True
)
print('✅ Indexes copied')

# Fix paths in indexes to point to local Colab
print('Fixing index paths...')
for idx_path in Path(f'{COLAB}/indexes').glob('*.json'):
    idx = json.loads(idx_path.read_text())
    for entry in idx:
        if 'thermal_path' in entry:
            entry['thermal_path'] = entry[
                'thermal_path'
            ].replace(
                'data/wildfire/patches',
                f'{COLAB}/wildfire/patches'
            ).replace(
                'data/solar/patches',
                f'{COLAB}/solar/patches'
            )
        if entry.get('s2_path'):
            entry['s2_path'] = entry[
                's2_path'
            ].replace(
                'data/wildfire/patches',
                f'{COLAB}/wildfire/patches'
            )
    idx_path.write_text(json.dumps(idx))
    print(f'  ✅ Fixed: {idx_path.name}')

# Verify disk
print('\nDisk status:')
os.system('df -h /content | tail -1')
os.system(f'du -sh {COLAB}')

# Verify datasets
VENV_PYTHON = '/content/venv311/bin/python3'
r = subprocess.run([VENV_PYTHON, '-c', f'''
import os, sys
os.environ["MPLBACKEND"] = "agg"
sys.path.insert(0, "/content/thermalwatch")

from pathlib import Path
from src.models.training.finetune import WildfireDataset, SolarDataset
from src.models.training.ssl_pretrain import ThermalSSLDataset

INDEX = Path("{COLAB}/indexes")

wf_train  = WildfireDataset(INDEX / "wildfire_train_index.json", "train")
wf_val    = WildfireDataset(INDEX / "wildfire_val_index.json", "val")
sol_train = SolarDataset(INDEX / "solar_train_index.json", "train")
sol_val   = SolarDataset(INDEX / "solar_val_index.json", "val")
ssl_train = ThermalSSLDataset(
    INDEX / "wildfire_train_index.json",
    INDEX / "solar_train_index.json",
    split="train",
)

print(f"Wildfire train: {{len(wf_train)}} | val: {{len(wf_val)}}")
print(f"Solar train:    {{len(sol_train)}} | val: {{len(sol_val)}}")
print(f"SSL train:      {{len(ssl_train)}}")

# Test loading speed
import time
from torch.utils.data import DataLoader
loader = DataLoader(wf_train, batch_size=32, num_workers=4)
t0 = time.time()
for i, batch in enumerate(loader):
    if i >= 4:
        break
elapsed = time.time() - t0
print(f"Loading speed: 5 batches in {{elapsed:.1f}}s")
print("✅ All datasets verified")
'''], capture_output=True, text=True)
print(r.stdout)
if r.stderr:
    print('STDERR:', r.stderr[-200:])

Copying wildfire patches (~29GB)...
✅ Wildfire patches copied
Copying solar patches (~19GB)...
✅ Solar patches copied
Copying indexes...
✅ Indexes copied
Fixing index paths...
  ✅ Fixed: wildfire_val_index.json
  ✅ Fixed: wildfire_train_index.json
  ✅ Fixed: solar_train_index.json
  ✅ Fixed: solar_val_index.json

Disk status:
Wildfire train: 81328 | val: 28500
Solar train:    166861 | val: 53699
SSL train:      248189
Loading speed: 5 batches in 3.2s
✅ All datasets verified



In [9]:
# Cell 5b: Copy area indexes + fix all paths
import shutil, json
from pathlib import Path

GDRIVE = '/gdrive/MyDrive/thermalwatch'
COLAB  = '/content/thermalwatch/data'

# Copy per-area indexes
print('Copying wildfire area indexes...')
for f in Path(f'{GDRIVE}/data/wildfire/indexes').glob('*_index.json'):
    shutil.copy(f, f'{COLAB}/wildfire/patches/{f.name}')
    print(f'  ✅ {f.name}')

print('Copying solar area indexes...')
for f in Path(f'{GDRIVE}/data/solar/indexes').glob('*_index.json'):
    shutil.copy(f, f'{COLAB}/solar/patches/{f.name}')
    print(f'  ✅ {f.name}')

# Fix paths in per-area indexes
print('\nFixing wildfire area index paths...')
for idx_path in Path(f'{COLAB}/wildfire/patches').glob('*_index.json'):
    idx = json.loads(idx_path.read_text())
    for entry in idx:
        if 'thermal_path' in entry:
            entry['thermal_path'] = entry['thermal_path'].replace(
                'data/wildfire/patches',
                f'{COLAB}/wildfire/patches'
            )
        if entry.get('s2_path'):
            entry['s2_path'] = entry['s2_path'].replace(
                'data/wildfire/patches',
                f'{COLAB}/wildfire/patches'
            )
    idx_path.write_text(json.dumps(idx))
    with_s2 = sum(
        1 for p in idx
        if p.get('s2_path') and Path(p['s2_path']).exists()
    )
    print(f'  ✅ {idx_path.name}: {with_s2} patches with S2')

print('\n✅ Area indexes ready for Prithvi extraction')

Copying wildfire area indexes...
  ✅ norcal_2020_index.json
  ✅ norcal_2021_index.json
  ✅ norcal_2022_index.json
  ✅ norcal_2023_index.json
  ✅ sierra_nevada_2020_index.json
  ✅ sierra_nevada_2021_index.json
  ✅ sierra_nevada_2023_index.json
  ✅ sierra_nevada_2022_index.json
  ✅ socal_2020_index.json
  ✅ socal_2021_index.json
  ✅ socal_2022_index.json
  ✅ socal_2023_index.json
Copying solar area indexes...
  ✅ phoenix_metro_2019_index.json
  ✅ phoenix_metro_2020_index.json
  ✅ phoenix_metro_2021_index.json
  ✅ phoenix_metro_2022_index.json
  ✅ phoenix_metro_2023_index.json
  ✅ sonoran_desert_2019_index.json
  ✅ sonoran_desert_2020_index.json
  ✅ sonoran_desert_2021_index.json
  ✅ sonoran_desert_2022_index.json
  ✅ sonoran_desert_2023_index.json
  ✅ tucson_2019_index.json
  ✅ tucson_2020_index.json
  ✅ tucson_2021_index.json
  ✅ tucson_2022_index.json
  ✅ tucson_2023_index.json

Fixing wildfire area index paths...
  ✅ norcal_2022_index.json: 0 patches with S2
  ✅ sierra_nevada_2020_ind

In [10]:
# Cell 5c: Verify all data needed for training
import json, os
from pathlib import Path

COLAB = '/content/thermalwatch/data'

print('=== TRAINING DATA VERIFICATION ===\n')

# 1. Check indexes
print('[1] Train/Val Indexes:')
for name in [
    'wildfire_train_index.json',
    'wildfire_val_index.json',
    'solar_train_index.json',
    'solar_val_index.json',
]:
    path = Path(f'{COLAB}/indexes/{name}')
    if path.exists():
        idx = json.loads(path.read_text())
        with_weather = sum(1 for p in idx if p.get('weather_features'))
        with_labels  = sum(1 for p in idx if p.get('risk_score') is not None or p.get('efficiency_score') is not None)
        print(f'  ✅ {name}: {len(idx)} patches | weather={with_weather/len(idx)*100:.1f}% | labels={with_labels/len(idx)*100:.1f}%')
    else:
        print(f'  ❌ {name}: MISSING')

# 2. Check wildfire patches
print('\n[2] Wildfire Patches:')
wf_patch_dir = Path(f'{COLAB}/wildfire/patches')
total_thermal = total_s2 = 0
for area_dir in sorted(wf_patch_dir.iterdir()):
    if not area_dir.is_dir():
        continue
    thermal = len(list(area_dir.glob('thermal_*.npy')))
    s2      = len(list(area_dir.glob('s2_*.npy')))
    total_thermal += thermal
    total_s2      += s2
    print(f'  {area_dir.name}: thermal={thermal} s2={s2}')
print(f'  TOTAL: thermal={total_thermal} s2={total_s2}')

# 3. Check solar patches
print('\n[3] Solar Patches:')
sol_patch_dir = Path(f'{COLAB}/solar/patches')
total_thermal = 0
for area_dir in sorted(sol_patch_dir.iterdir()):
    if not area_dir.is_dir():
        continue
    thermal = len(list(area_dir.glob('thermal_*.npy')))
    total_thermal += thermal
    print(f'  {area_dir.name}: thermal={thermal}')
print(f'  TOTAL: thermal={total_thermal}')

# 4. Check disk
print('\n[4] Disk Status:')
os.system('df -h /content | tail -1')

# 5. Check GPU
print('\n[5] GPU Status:')
os.system('nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader')

print('\n✅ Verification complete')

=== TRAINING DATA VERIFICATION ===

[1] Train/Val Indexes:
  ✅ wildfire_train_index.json: 81328 patches | weather=95.1% | labels=100.0%
  ✅ wildfire_val_index.json: 28500 patches | weather=97.2% | labels=100.0%
  ✅ solar_train_index.json: 166861 patches | weather=93.4% | labels=100.0%
  ✅ solar_val_index.json: 53699 patches | weather=94.0% | labels=100.0%

[2] Wildfire Patches:
  norcal_2020: thermal=4636 s2=4564
  norcal_2021: thermal=4666 s2=0
  norcal_2022: thermal=4104 s2=0
  norcal_2023: thermal=4803 s2=0
  sierra_nevada_2020: thermal=4639 s2=4488
  sierra_nevada_2021: thermal=4708 s2=2254
  sierra_nevada_2022: thermal=4895 s2=0
  sierra_nevada_2023: thermal=4923 s2=0
  socal_2020: thermal=4646 s2=4587
  socal_2021: thermal=4903 s2=0
  socal_2022: thermal=4855 s2=0
  socal_2023: thermal=4918 s2=0
  TOTAL: thermal=56696 s2=15893

[3] Solar Patches:
  phoenix_metro_2019: thermal=6544
  phoenix_metro_2020: thermal=6449
  phoenix_metro_2021: thermal=6487
  phoenix_metro_2022: thermal=

In [11]:
# Cell 5d: Copy new train/val indexes from GDrive to Colab
import shutil, json
from pathlib import Path

GDRIVE = '/gdrive/MyDrive/thermalwatch'
COLAB  = '/content/thermalwatch/data'

print('Copying new train/val indexes...')
for name in [
    'wildfire_train_index.json',
    'wildfire_val_index.json',
    'solar_train_index.json',
    'solar_val_index.json',
]:
    src = Path(f'{GDRIVE}/data/indexes/{name}')
    dst = Path(f'{COLAB}/indexes/{name}')
    shutil.copy(src, dst)
    print(f'  ✅ {name}: {dst.stat().st_size/1e6:.1f}MB')

# Fix paths to local Colab
print('\nFixing paths...')
for name in [
    'wildfire_train_index.json',
    'wildfire_val_index.json',
]:
    idx_path = Path(f'{COLAB}/indexes/{name}')
    idx      = json.loads(idx_path.read_text())
    for entry in idx:
        if 'thermal_path' in entry:
            entry['thermal_path'] = entry[
                'thermal_path'
            ].replace(
                'data/wildfire/patches',
                f'{COLAB}/wildfire/patches'
            )
        if entry.get('s2_path'):
            entry['s2_path'] = entry['s2_path'].replace(
                'data/wildfire/patches',
                f'{COLAB}/wildfire/patches'
            )
        if entry.get('prithvi_path'):
            entry['prithvi_path'] = entry[
                'prithvi_path'
            ].replace(
                'data/wildfire/patches',
                f'{COLAB}/wildfire/patches'
            )
    idx_path.write_text(json.dumps(idx))
    print(f'  ✅ Fixed: {name}')

for name in [
    'solar_train_index.json',
    'solar_val_index.json',
]:
    idx_path = Path(f'{COLAB}/indexes/{name}')
    idx      = json.loads(idx_path.read_text())
    for entry in idx:
        if 'thermal_path' in entry:
            entry['thermal_path'] = entry[
                'thermal_path'
            ].replace(
                'data/solar/patches',
                f'{COLAB}/solar/patches'
            )
    idx_path.write_text(json.dumps(idx))
    print(f'  ✅ Fixed: {name}')

# Verify
print('\nVerification:')
for name in [
    'wildfire_train_index.json',
    'wildfire_val_index.json',
    'solar_train_index.json',
    'solar_val_index.json',
]:
    idx  = json.loads(
        Path(f'{COLAB}/indexes/{name}').read_text()
    )
    p    = idx[0]
    exists = Path(p['thermal_path']).exists()
    print(
        f'  {name}: {len(idx)} patches | '
        f'thermal exists: {exists}'
    )

print('\n✅ New indexes ready')

Copying new train/val indexes...
  ✅ wildfire_train_index.json: 55.4MB
  ✅ wildfire_val_index.json: 18.5MB
  ✅ solar_train_index.json: 109.8MB
  ✅ solar_val_index.json: 35.4MB

Fixing paths...
  ✅ Fixed: wildfire_train_index.json
  ✅ Fixed: wildfire_val_index.json
  ✅ Fixed: solar_train_index.json
  ✅ Fixed: solar_val_index.json

Verification:
  wildfire_train_index.json: 81328 patches | thermal exists: True
  wildfire_val_index.json: 28500 patches | thermal exists: True
  solar_train_index.json: 166861 patches | thermal exists: True
  solar_val_index.json: 53699 patches | thermal exists: True

✅ New indexes ready


In [9]:
# Cell 6: Extract Prithvi embeddings for wildfire patches
import subprocess, os

VENV_PYTHON = '/content/venv311/bin/python3'
COLAB       = '/content/thermalwatch/data'

r = subprocess.Popen([
    VENV_PYTHON, '-m',
    'src.data.extract_prithvi_embeddings',
    '--task',       'wildfire',
    '--data-dir',   f'{COLAB}/wildfire',
    '--batch-size', '64',
], env={**os.environ,
        'PYTHONPATH': '/content/thermalwatch',
        'MPLBACKEND': 'agg'},
   stdout=subprocess.PIPE,
   stderr=subprocess.STDOUT,
   text=True, bufsize=1,
)

for line in r.stdout:
    print(line, end='', flush=True)

r.wait()
print(f'\nExited: {r.returncode}')

2026-09-18 23:49:28,515 INFO Device: cuda
2026-09-18 23:49:28,515 INFO Loading Prithvi-EO-2.0-300M...
2026-09-18 23:49:39,866 INFO model_bands not passed. Assuming bands are ordered in the same way as [<HLSBands.BLUE: 'BLUE'>, <HLSBands.GREEN: 'GREEN'>, <HLSBands.RED: 'RED'>, <HLSBands.NIR_NARROW: 'NIR_NARROW'>, <HLSBands.SWIR_1: 'SWIR_1'>, <HLSBands.SWIR_2: 'SWIR_2'>].Pretrained patch_embed layer may be misaligned with current bands
2026-09-18 23:49:49,962 INFO Loaded weights for HLSBands.BLUE in position 0 of patch embed
2026-09-18 23:49:49,971 INFO Loaded weights for HLSBands.GREEN in position 1 of patch embed
2026-09-18 23:49:49,971 INFO Loaded weights for HLSBands.RED in position 2 of patch embed
2026-09-18 23:49:49,972 INFO Loaded weights for HLSBands.NIR_NARROW in position 3 of patch embed
2026-09-18 23:49:49,972 INFO Loaded weights for HLSBands.SWIR_1 in position 4 of patch embed
2026-09-18 23:49:49,972 INFO Loaded weights for HLSBands.SWIR_2 in position 5 of patch embed
2026-0

In [10]:
# Cell 6b: Update train/val indexes with prithvi_path
import json
from pathlib import Path

COLAB = '/content/thermalwatch/data'

print('Building prithvi_path lookup...')
prithvi_lookup = {}
for idx_path in Path(f'{COLAB}/wildfire/patches').glob('*_index.json'):
    idx = json.loads(idx_path.read_text())
    for entry in idx:
        if entry.get('prithvi_path') and Path(entry['prithvi_path']).exists():
            prithvi_lookup[entry['patch_id']] = entry['prithvi_path']

print(f'Total prithvi embeddings: {len(prithvi_lookup)}')

for split in ['wildfire_train_index.json', 'wildfire_val_index.json']:
    idx_path = Path(f'{COLAB}/indexes/{split}')
    idx      = json.loads(idx_path.read_text())
    updated  = 0
    for entry in idx:
        pid = entry['patch_id']
        if pid in prithvi_lookup:
            entry['prithvi_path'] = prithvi_lookup[pid]
            updated += 1
    idx_path.write_text(json.dumps(idx))
    print(f'✅ {split}: {updated}/{len(idx)} entries updated')

print('\n✅ Indexes updated with prithvi_path')

Building prithvi_path lookup...
Total prithvi embeddings: 15829
✅ wildfire_train_index.json: 31277/81328 entries updated
✅ wildfire_val_index.json: 0/28500 entries updated

✅ Indexes updated with prithvi_path


In [ ]:
# Cell 7: InfoNCE SSL v3
import subprocess, os

VENV_PYTHON = '/content/venv311/bin/python3'
COLAB       = '/content/thermalwatch/data'
CKPT_DIR    = '/gdrive/MyDrive/thermalwatch/checkpoints/ssl'
os.makedirs(CKPT_DIR, exist_ok=True)

r = subprocess.Popen([
    VENV_PYTHON, '-m', 'src.models.training.ssl_pretrain',
    '--index-dir',   f'{COLAB}/indexes',
    '--ckpt-dir',    CKPT_DIR,
    '--epochs',      '100',
    '--batch-size',  '128',
    '--lr-max',      '3e-4',
    '--lr-min',      '1e-6',
    '--warmup',      '10',
    '--patience',    '10',
    '--temperature', '0.5',
], env={**os.environ,
        'PYTHONPATH': '/content/thermalwatch',
        'MPLBACKEND': 'agg'},
   stdout=subprocess.PIPE,
   stderr=subprocess.STDOUT,
   text=True, bufsize=1,
)

for line in r.stdout:
    print(line, end='', flush=True)

r.wait()
print(f'\nExited: {r.returncode}')

In [11]:
# Cell 8: Wildfire Fine-tuning
import subprocess, os

VENV_PYTHON = '/content/venv311/bin/python3'
COLAB       = '/content/thermalwatch/data'
CKPT_DIR    = '/gdrive/MyDrive/thermalwatch/checkpoints/finetune/wildfire'
os.makedirs(CKPT_DIR, exist_ok=True)

r = subprocess.Popen([
    VENV_PYTHON, '-m', 'src.models.training.finetune',
    '--task',          'wildfire',
    '--index-dir',     f'{COLAB}/indexes',
    '--ckpt-dir',      CKPT_DIR,
    '--batch-size',    '256',
    '--phase1-epochs', '30',
    '--phase2-epochs', '25',
    '--lr-head',       '1e-3',
    '--lr-backbone',   '1e-5',
    '--patience',      '5',
], env={**os.environ,
        'PYTHONPATH': '/content/thermalwatch',
        'MPLBACKEND': 'agg',
        'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True',
       },
   stdout=subprocess.PIPE,
   stderr=subprocess.STDOUT,
   text=True, bufsize=1,
)

for line in r.stdout:
    print(line, end='', flush=True)

r.wait()
print(f'\nExited: {r.returncode}')

2026-09-18 00:05:14,265 INFO Device: cuda | Task: wildfire
2026-09-18 00:05:15,267 INFO WildfireDataset train: 81328 | S2=33.9%
2026-09-18 00:05:15,665 INFO WildfireDataset val: 28500 | S2=0.0%
2026-09-18 00:05:18,051 INFO ThermalEncoder: 1 channel → 512d
2026-09-18 00:05:25,718 INFO model_bands not passed. Assuming bands are ordered in the same way as [<HLSBands.BLUE: 'BLUE'>, <HLSBands.GREEN: 'GREEN'>, <HLSBands.RED: 'RED'>, <HLSBands.NIR_NARROW: 'NIR_NARROW'>, <HLSBands.SWIR_1: 'SWIR_1'>, <HLSBands.SWIR_2: 'SWIR_2'>].Pretrained patch_embed layer may be misaligned with current bands
2026-09-18 00:05:35,705 INFO Loaded weights for HLSBands.BLUE in position 0 of patch embed
2026-09-18 00:05:35,706 INFO Loaded weights for HLSBands.GREEN in position 1 of patch embed
2026-09-18 00:05:35,706 INFO Loaded weights for HLSBands.RED in position 2 of patch embed
2026-09-18 00:05:35,707 INFO Loaded weights for HLSBands.NIR_NARROW in position 3 of patch embed
2026-09-18 00:05:35,707 INFO Loaded we

In [11]:
# Cell 9: Solar Fine-tuning — large batch
import subprocess, os

VENV_PYTHON = '/content/venv311/bin/python3'
COLAB       = '/content/thermalwatch/data'
CKPT_DIR    = '/gdrive/MyDrive/thermalwatch/checkpoints/finetune/solar'
os.makedirs(CKPT_DIR, exist_ok=True)

r = subprocess.Popen([
    VENV_PYTHON, '-m', 'src.models.training.finetune',
    '--task',          'solar',
    '--index-dir',     f'{COLAB}/indexes',
    '--ckpt-dir',      CKPT_DIR,
    '--batch-size',    '1024',
    '--phase1-epochs', '30',
    '--phase2-epochs', '25',
    '--lr-head',       '1e-3',
    '--lr-backbone',   '1e-5',
    '--patience',      '5',
], env={**os.environ,
        'PYTHONPATH': '/content/thermalwatch',
        'MPLBACKEND': 'agg',
        'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True',
       },
   stdout=subprocess.PIPE,
   stderr=subprocess.STDOUT,
   text=True, bufsize=1,
)

for line in r.stdout:
    print(line, end='', flush=True)

r.wait()
print(f'\nExited: {r.returncode}')

2026-09-19 00:10:11,439 INFO Device: cuda | Task: solar
2026-09-19 00:10:13,692 INFO SolarDataset train: 166861
2026-09-19 00:10:14,449 INFO SolarDataset val: 53699
2026-09-19 00:10:17,016 INFO ThermalEncoder: 1 channel → 512d
2026-09-19 00:10:24,876 INFO model_bands not passed. Assuming bands are ordered in the same way as [<HLSBands.BLUE: 'BLUE'>, <HLSBands.GREEN: 'GREEN'>, <HLSBands.RED: 'RED'>, <HLSBands.NIR_NARROW: 'NIR_NARROW'>, <HLSBands.SWIR_1: 'SWIR_1'>, <HLSBands.SWIR_2: 'SWIR_2'>].Pretrained patch_embed layer may be misaligned with current bands
2026-09-19 00:10:34,449 INFO Loaded weights for HLSBands.BLUE in position 0 of patch embed
2026-09-19 00:10:34,449 INFO Loaded weights for HLSBands.GREEN in position 1 of patch embed
2026-09-19 00:10:34,450 INFO Loaded weights for HLSBands.RED in position 2 of patch embed
2026-09-19 00:10:34,450 INFO Loaded weights for HLSBands.NIR_NARROW in position 3 of patch embed
2026-09-19 00:10:34,450 INFO Loaded weights for HLSBands.SWIR_1 in 

In [12]:
# Cell 10: Evaluate both models
import subprocess, os

VENV_PYTHON = '/content/venv311/bin/python3'
COLAB       = '/content/thermalwatch/data'
GDRIVE      = '/gdrive/MyDrive/thermalwatch'

r = subprocess.run([VENV_PYTHON, '-c', f'''
import os, sys, torch
import numpy as np
os.environ["MPLBACKEND"] = "agg"
sys.path.insert(0, "/content/thermalwatch")

from pathlib import Path
from torch.utils.data import DataLoader
from src.models.backbone.thermal_backbone import ThermalWatchBackbone
from src.models.training.finetune import (
    WildfireDataset, SolarDataset,
    WildfireHead, SolarHead,
    compute_wildfire_metrics, compute_solar_metrics,
)

COLAB  = "{COLAB}"
GDRIVE = "{GDRIVE}"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def evaluate(task, ckpt_path, dataset, loader):
    backbone = ThermalWatchBackbone(
        pretrained_optical=False
    ).to(device)
    head = (
        WildfireHead(512) if task == "wildfire"
        else SolarHead(512)
    ).to(device)

    if not Path(ckpt_path).exists():
        print(f"❌ Checkpoint not found: {{ckpt_path}}")
        return

    ckpt = torch.load(ckpt_path, map_location=device)
    backbone.load_state_dict(ckpt["backbone_state"])
    head.load_state_dict(ckpt["head_state"])
    backbone.eval()
    head.eval()

    all_preds   = []
    all_targets = []

    with torch.no_grad():
        for batch in loader:
            thermal = batch["thermal"].to(device)
            optical = batch["optical"].to(device)
            weather = batch["weather"].to(device)
            osm     = batch["osm"].to(device)
            has_s2  = batch["has_s2"]
            mask    = has_s2.to(device).float()
            optical = optical * mask.view(-1, 1, 1, 1)

            emb   = backbone(
                thermal=thermal,
                optical=optical if has_s2.any() else None,
                weather=weather,
                osm=osm,
            )
            preds = head(emb)
            all_preds.append(
                {{k: v.detach() for k, v in preds.items()}}
            )
            all_targets.append(
                {{k: v.detach().cpu().numpy()
                 for k, v in batch["targets"].items()}}
            )

    if task == "wildfire":
        metrics = compute_wildfire_metrics(all_preds, all_targets)
        print(f"\\n=== WILDFIRE RESULTS ===")
        print(f"risk_score:     RMSE={{metrics['risk_rmse']:.4f}} R²={{metrics['risk_r2']:.4f}} MAE={{metrics['risk_mae']:.4f}}")
        print(f"alert_level:    Acc={{metrics['alert_acc']*100:.1f}}% F1={{metrics['alert_f1']:.4f}}")
        print(f"spread_prob:    RMSE={{metrics['spread_rmse']:.4f}} R²={{metrics['spread_r2']:.4f}}")
        print(f"structure_risk: RMSE={{metrics['struct_rmse']:.4f}} R²={{metrics['struct_r2']:.4f}}")
    else:
        metrics = compute_solar_metrics(all_preds, all_targets)
        print(f"\\n=== SOLAR RESULTS ===")
        print(f"efficiency:  RMSE={{metrics['eff_rmse']:.4f}} R²={{metrics['eff_r2']:.4f}} MAE={{metrics['eff_mae']:.4f}}")
        print(f"hotspot:     RMSE={{metrics['hot_rmse']:.4f}} R²={{metrics['hot_r2']:.4f}}")
        print(f"degradation: RMSE={{metrics['deg_rmse']:.4f}} R²={{metrics['deg_r2']:.4f}}")
        print(f"maintenance: Acc={{metrics['maint_acc']*100:.1f}}% F1={{metrics['maint_f1']:.4f}} Prec={{metrics['maint_prec']:.4f}} Rec={{metrics['maint_rec']:.4f}}")

INDEX = Path(f"{{COLAB}}/indexes")

# Evaluate wildfire
print("Evaluating wildfire model...")
wf_val    = WildfireDataset(INDEX / "wildfire_val_index.json", "val")
wf_loader = DataLoader(wf_val, batch_size=32, num_workers=2)
evaluate(
    "wildfire",
    f"{{GDRIVE}}/checkpoints/finetune/wildfire/wildfire_best_phase1.pt",
    wf_val, wf_loader,
)

# Evaluate solar
print("\\nEvaluating solar model...")
sol_val    = SolarDataset(INDEX / "solar_val_index.json", "val")
sol_loader = DataLoader(sol_val, batch_size=32, num_workers=2)
evaluate(
    "solar",
    f"{{GDRIVE}}/checkpoints/finetune/solar/solar_best_phase1.pt",
    sol_val, sol_loader,
)
'''], capture_output=True, text=True)
print(r.stdout)
if r.stderr:
    print('STDERR:', r.stderr[-500:])

Evaluating wildfire model...

=== WILDFIRE RESULTS ===
risk_score:     RMSE=0.3142 R²=-0.9743 MAE=0.2846
alert_level:    Acc=75.6% F1=0.2314
spread_prob:    RMSE=0.0683 R²=-0.0500
structure_risk: RMSE=0.0300 R²=-0.4899

Evaluating solar model...

=== SOLAR RESULTS ===
efficiency:  RMSE=0.2123 R²=0.2640 MAE=0.1693
hotspot:     RMSE=0.1171 R²=-0.4728
degradation: RMSE=0.0944 R²=0.1809
maintenance: Acc=95.7% F1=0.0679 Prec=0.0365 Rec=0.4855

